# MMALS Prototype v1.1 RC2O — Axis A Host Specialization Audit — UPDATED

**Notebook:** `MMALS_Prototype_1_1_RC2O_AXIS_A_Host_Specialization_Audit_Colab_UPDATED`  
**Status:** diagnostic / audit branch, not a new policy branch.  
**Frozen candidate:** RC2O must be treated as read-only during this audit.

## What changed in this updated notebook

This version turns the previous demo-smoke audit into a **real RC2O evidence-ready harness**:

1. Defaults are now conservative for real evidence: `MODEL_SOURCE = 'checkpoint'`, `RUN_MODE = 'evidence'`, and `SEEDS = [0, 1, 2]`.
2. Demo mode is still available, but it is explicitly gated by `ALLOW_DEMO_SMOKE = True`; demo results are never interpreted as RC2O evidence.
3. The LaTeX markdown issue that broke PDF export has been removed by using nbconvert-safe math notation.
4. Cross-seed specialization alignment is now part of the recommended evidence workflow.
5. A Google Drive publication step is included at the end. When run in Colab, the notebook copies the ZIP, manifest, tables, plots, and summary report into Drive.

## Research question

Do MMALS hosts become **functionally specialized**, or are they only redundant subnetworks?

## Conservative claim level

This notebook does **not** claim that MMALS proves biological specialization.

It tests whether a frozen MMALS candidate exhibits **measurable functional host specialization** when several independent signals agree:

1. **Routing specialization:** entropy decreases and dominance increases.
2. **Host usefulness:** mutualistic gain is positive.
3. **Functional contribution:** accuracy / CE / output drift degrade when a useful host is removed.
4. **Representation separation:** host latent spaces are distinguishable.
5. **Cross-seed stability:** similar functional specialization emerges across seeds, allowing host-label permutation.
6. **Non-triviality:** specialization is not explained only by parameter count or post-hoc route randomness.

## Core rule

**RC2O is frozen. Axis A only performs post-hoc interventions and diagnostics.**

No selector, router, guard, head policy, temperature, or loss coefficient should be changed in the frozen checkpoint.


## Axis A evidence map

| Evidence type | Metric | Interpretation |
|---|---:|---|
| Routing specialization | entropy ↓, dominance ↑ | The router is not broadcasting everywhere. |
| Host usefulness | positive mutualistic gain | Removing host `i` hurts the local objective. |
| Functional contribution | ΔAccuracy, ΔCE, output KL, latent MSE | The host causally affects output behavior. |
| Representation separation | linear CKA / cosine centroid separation | Hosts do not compute identical latent spaces. |
| Stability | cross-seed aligned route/gain patterns | Specialization is reproducible modulo host permutation. |
| Non-triviality | collapse controls fail | Learned host-route structure matters beyond capacity. |

## Four tests implemented

1. **Host ablation test** — remove host `i`, renormalize route, measure accuracy loss, CE increase, latent drift, output drift.
2. **Host swap test** — swap context routes; if performance collapses, host-context binding is meaningful.
3. **Host collapse test** — force uniform / shuffled / global-single-host routing; RC2O should lose the specialization advantage.
4. **Cross-seed specialization test** — align host IDs across seeds and test whether equivalent specialization patterns recur.


In [ ]:
# ============================================================
# 0. Optional Colab setup
# ============================================================
# In most Colab runtimes, torch, torchvision, pandas, numpy and matplotlib are installed.
# Uncomment only if needed.
# !pip -q install torchvision pandas numpy matplotlib scikit-learn scipy

from __future__ import annotations

import os
import io
import gc
import json
import math
import time
import copy
import zipfile
import random
import hashlib
import shutil
import inspect
import itertools
import warnings
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Any, Dict, List, Tuple, Optional, Callable

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

try:
    from torchvision import datasets, transforms
    TORCHVISION_AVAILABLE = True
except Exception as e:
    TORCHVISION_AVAILABLE = False
    print('torchvision unavailable:', repr(e))

try:
    from IPython.display import display
except Exception:
    display = print

try:
    from scipy.optimize import linear_sum_assignment
    SCIPY_AVAILABLE = True
except Exception:
    SCIPY_AVAILABLE = False

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('DEVICE:', DEVICE)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# ============================================================
# 1. Configuration — evidence-ready defaults
# ============================================================

# Recommended workflow:
# 1) First run with MODEL_SOURCE='demo_smoke' and ALLOW_DEMO_SMOKE=True only to verify runtime.
# 2) Then switch back to MODEL_SOURCE='checkpoint' for any scientific claim.
# 3) Keep RUN_MODE='evidence' for the first real RC2O audit.

RUN_MODE = 'evidence'  # 'smoke', 'evidence', 'paper'
DATASET_NAME = 'FashionMNIST'  # 'MNIST' or 'FashionMNIST'
PAIR_MODE = 'overlap_chain'  # 'disjoint_pairs' or 'overlap_chain'

# Frozen policy candidate metadata. This is for the audit manifest.
FROZEN_POLICY_NAME = 'RC2O'
NOTEBOOK_NAME = 'MMALS_Prototype_1_1_RC2O_AXIS_A_Host_Specialization_Audit_Colab_UPDATED'

# Scientific-evidence default.
# Any result with MODEL_SOURCE='demo_smoke' is pipeline-validation only, never RC2O evidence.
MODEL_SOURCE = 'checkpoint'  # 'checkpoint' or 'demo_smoke'
ALLOW_DEMO_SMOKE = False

# Real RC2O checkpoint path.
# Replace with your frozen RC2O checkpoint in Google Drive.
FROZEN_CKPT_PATH = '/content/drive/MyDrive/MMALS/checkpoints/RC2O_frozen.pt'

# Optional: one checkpoint per seed for cross-seed evidence.
# Use either:
#   SEED_CKPT_PATHS = {0: '...', 1: '...', 2: '...'}
# or leave empty if your single checkpoint contains the selected seeds or you only audit seed 0.
SEED_CKPT_PATHS: Dict[int, str] = {
    # 0: '/content/drive/MyDrive/MMALS/checkpoints/RC2O_seed0.pt',
    # 1: '/content/drive/MyDrive/MMALS/checkpoints/RC2O_seed1.pt',
    # 2: '/content/drive/MyDrive/MMALS/checkpoints/RC2O_seed2.pt',
}

# Results and Google Drive publication.
RESULTS_DIR = Path('./mmals_v11_rc2o_axis_a_host_specialization_audit')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR = RESULTS_DIR / 'plots'
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR = RESULTS_DIR / 'tables'
TABLES_DIR.mkdir(parents=True, exist_ok=True)

PUBLISH_TO_DRIVE = True
DRIVE_OUTPUT_ROOT = '/content/drive/MyDrive/MMALS/RC2O/Axis_A_Host_Specialization_Audit'
COPY_NOTEBOOK_TO_DRIVE = True

if RUN_MODE == 'smoke':
    TRAIN_PER_CLASS = 200
    TEST_PER_CLASS = 100
    SEEDS = [0]
    EPOCHS_PER_TASK = 2
    BATCH_SIZE = 128
    MEMORY_PER_TASK = 96
elif RUN_MODE == 'evidence':
    TRAIN_PER_CLASS = 1200
    TEST_PER_CLASS = 400
    SEEDS = [0, 1, 2]
    EPOCHS_PER_TASK = 8
    BATCH_SIZE = 128
    MEMORY_PER_TASK = 256
elif RUN_MODE == 'paper':
    TRAIN_PER_CLASS = 3000
    TEST_PER_CLASS = 1000
    SEEDS = list(range(10))
    EPOCHS_PER_TASK = 15
    BATCH_SIZE = 128
    MEMORY_PER_TASK = 512
else:
    raise ValueError("RUN_MODE must be 'smoke', 'evidence', or 'paper'")

N_TASKS = 5
N_HOSTS = 5
INPUT_DIM = 28 * 28
HIDDEN_DIM = 256
LATENT_DIM = 64
ROUTE_HIDDEN_DIM = 128
TASK_EMB_DIM = 16
LR = 1e-3
TEMPERATURE = 0.7
DISTILL_TEMPERATURE = 2.0

# Axis A thresholds. Adjust only before running, then freeze in manifest.
ENTROPY_SPECIALIZED_MAX = math.log(N_HOSTS) * 0.70
DOMINANCE_SPECIALIZED_MIN = 0.45
GAIN_POSITIVE_EPS = 1e-4
ACC_DROP_IMPORTANT = 0.01
CE_INCREASE_IMPORTANT = 0.01
OUTPUT_KL_IMPORTANT = 1e-3
LATENT_DRIFT_IMPORTANT = 1e-4
CKA_SEPARATION_MIN = 0.05

# Safety gates.
if MODEL_SOURCE == 'demo_smoke' and not ALLOW_DEMO_SMOKE:
    raise RuntimeError(
        "MODEL_SOURCE='demo_smoke' is disabled. Set ALLOW_DEMO_SMOKE=True only for pipeline testing. "
        "For RC2O evidence, keep MODEL_SOURCE='checkpoint'."
    )
if MODEL_SOURCE == 'demo_smoke' and RUN_MODE != 'smoke':
    raise RuntimeError("Demo mode is allowed only with RUN_MODE='smoke'.")

print(json.dumps({
    'RUN_MODE': RUN_MODE,
    'MODEL_SOURCE': MODEL_SOURCE,
    'FROZEN_POLICY_NAME': FROZEN_POLICY_NAME,
    'DATASET_NAME': DATASET_NAME,
    'PAIR_MODE': PAIR_MODE,
    'TRAIN_PER_CLASS': TRAIN_PER_CLASS,
    'TEST_PER_CLASS': TEST_PER_CLASS,
    'SEEDS': SEEDS,
    'EPOCHS_PER_TASK': EPOCHS_PER_TASK,
    'BATCH_SIZE': BATCH_SIZE,
    'RESULTS_DIR': str(RESULTS_DIR),
    'PUBLISH_TO_DRIVE': PUBLISH_TO_DRIVE,
    'DRIVE_OUTPUT_ROOT': DRIVE_OUTPUT_ROOT,
}, indent=2))


In [ ]:
# ============================================================
# 2. Reproducibility and utility helpers
# ============================================================

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def clear_memory() -> None:
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def make_loader(ds, batch_size: int = BATCH_SIZE, shuffle: bool = False) -> DataLoader:
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, num_workers=0, pin_memory=False)


def count_parameters(model: nn.Module, trainable_only: bool = True) -> int:
    if trainable_only:
        return sum(p.numel() for p in model.parameters() if p.requires_grad)
    return sum(p.numel() for p in model.parameters())


def model_size_mb(model: nn.Module, bytes_per_param: int = 4) -> float:
    return count_parameters(model, trainable_only=False) * bytes_per_param / (1024 ** 2)


def set_requires_grad(module_or_param, requires_grad: bool) -> None:
    """Enable/disable gradients for an nn.Module or a single torch.nn.Parameter.

    This helper is intentionally robust because previous MMALS PNN/host-freeze
    variants can crash when module freezing is applied inconsistently.
    """
    if isinstance(module_or_param, nn.Module):
        for p in module_or_param.parameters():
            p.requires_grad = requires_grad
        return
    if isinstance(module_or_param, torch.nn.Parameter):
        module_or_param.requires_grad = requires_grad
        return
    raise TypeError(f'Unsupported type for set_requires_grad: {type(module_or_param)!r}')


def file_sha256(path: str | Path) -> Optional[str]:
    p = Path(path)
    if not p.exists() or not p.is_file():
        return None
    h = hashlib.sha256()
    with p.open('rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()


def tensor_to_numpy(x: torch.Tensor) -> np.ndarray:
    return x.detach().cpu().float().numpy()


def safe_mean(values: List[float]) -> float:
    values = [float(v) for v in values if v is not None and np.isfinite(v)]
    return float(np.mean(values)) if values else float('nan')


def entropy_np(r: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    r = np.asarray(r, dtype=np.float64)
    return -np.sum(r * np.log(r + eps), axis=-1)


def dominance_np(r: np.ndarray) -> np.ndarray:
    return np.max(np.asarray(r), axis=-1)


def kl_probs_np(p: np.ndarray, q: np.ndarray, eps: float = 1e-8) -> np.ndarray:
    p = np.clip(p, eps, 1.0)
    q = np.clip(q, eps, 1.0)
    return np.sum(p * (np.log(p) - np.log(q)), axis=-1)


def normalize_route_tensor(r: torch.Tensor, eps: float = 1e-8) -> torch.Tensor:
    r = torch.clamp(r, min=0.0)
    return r / torch.clamp(r.sum(dim=-1, keepdim=True), min=eps)

## Dataset builder

The audit supports both:

- `disjoint_pairs`: `(0 vs 1), (2 vs 3), (4 vs 5), (6 vs 7), (8 vs 9)`
- `overlap_chain`: `(0 vs 1), (1 vs 2), (2 vs 3), (3 vs 4), (4 vs 5)`

For RC2O, keep the same dataset regime used when freezing the candidate. Do not change the regime after starting the audit.

In [ ]:
# ============================================================
# 3. Dataset builders
# ============================================================

if not TORCHVISION_AVAILABLE:
    raise RuntimeError('torchvision is required for MNIST/FashionMNIST dataset loading.')


def get_transform(name: str):
    return transforms.Compose([
        transforms.ToTensor(),
        transforms.Lambda(lambda x: x.view(-1)),
    ])


def load_base_dataset(name: str):
    transform = get_transform(name)
    if name == 'MNIST':
        train = datasets.MNIST('./data', train=True, download=True, transform=transform)
        test = datasets.MNIST('./data', train=False, download=True, transform=transform)
    elif name == 'FashionMNIST':
        train = datasets.FashionMNIST('./data', train=True, download=True, transform=transform)
        test = datasets.FashionMNIST('./data', train=False, download=True, transform=transform)
    else:
        raise ValueError("DATASET_NAME must be 'MNIST' or 'FashionMNIST'")
    return train, test


def get_labels(base_dataset) -> np.ndarray:
    if hasattr(base_dataset, 'targets'):
        return np.array(base_dataset.targets)
    raise ValueError('Dataset has no targets attribute.')


def task_pairs(pair_mode: str) -> List[Tuple[int, int]]:
    if pair_mode == 'disjoint_pairs':
        return [(0, 1), (2, 3), (4, 5), (6, 7), (8, 9)]
    if pair_mode == 'overlap_chain':
        return [(0, 1), (1, 2), (2, 3), (3, 4), (4, 5)]
    raise ValueError("PAIR_MODE must be 'disjoint_pairs' or 'overlap_chain'")


def subset_pair_dataset(base_dataset, a: int, b: int, max_per_class: Optional[int] = None, seed: int = 42) -> TensorDataset:
    rng = np.random.default_rng(seed)
    labels = get_labels(base_dataset)
    idx_a = np.where(labels == a)[0]
    idx_b = np.where(labels == b)[0]
    if max_per_class is not None:
        idx_a = rng.choice(idx_a, size=min(max_per_class, len(idx_a)), replace=False)
        idx_b = rng.choice(idx_b, size=min(max_per_class, len(idx_b)), replace=False)
    indices = np.concatenate([idx_a, idx_b])
    rng.shuffle(indices)
    xs, ys = [], []
    for idx in indices:
        x, y = base_dataset[int(idx)]
        xs.append(x.float())
        ys.append(1 if int(y) == b else 0)
    return TensorDataset(torch.stack(xs).float(), torch.tensor(ys).long())


def build_tasks(name: str = DATASET_NAME, pair_mode: str = PAIR_MODE, seed: int = 42) -> List[Dict[str, Any]]:
    train_base, test_base = load_base_dataset(name)
    pairs = task_pairs(pair_mode)
    tasks = []
    for task_id, (a, b) in enumerate(pairs):
        train_ds = subset_pair_dataset(train_base, a, b, max_per_class=TRAIN_PER_CLASS, seed=seed + task_id)
        test_ds = subset_pair_dataset(test_base, a, b, max_per_class=TEST_PER_CLASS, seed=seed + 100 + task_id)
        tasks.append({
            'task_id': task_id,
            'name': f'{a} vs {b}',
            'pair': (a, b),
            'train': train_ds,
            'test': test_ds,
        })
    return tasks

set_seed(SEEDS[0])
tasks = build_tasks(DATASET_NAME, PAIR_MODE, seed=42)
summary = [(t['task_id'], t['name'], len(t['train']), len(t['test'])) for t in tasks]
summary

## Frozen RC2O adapter contract

The audit needs a model output dictionary with these keys:

```python
{
    'logits': Tensor[B, C],
    'routes': Tensor[B, H] or Tensor[H],
    'z_fungal': Tensor[B, D],
    'transformed_hosts': Tensor[B, H, D],   # strongly recommended
    'host_latents': Tensor[B, H, D],        # optional
}
```

For the real frozen RC2O checkpoint, edit `build_frozen_rc2o_model()` and, if needed, `MMALSForwardAdapter.forward()`.

The included demo model is only a **pipeline smoke-test scaffold**. It is never scientific evidence for RC2O.


In [ ]:
# ============================================================
# 4. Demo MMALS-compatible model for smoke testing the audit
# ============================================================

class DemoFungalMMALS(nn.Module):
    """Small MMALS-compatible model used only when MODEL_SOURCE='demo_smoke'.

    This is not the RC2O architecture. Its purpose is to verify that the Axis A audit
    pipeline runs end-to-end: normal evaluation, ablation, swap, collapse, and exports.
    """
    def __init__(
        self,
        input_dim: int = INPUT_DIM,
        hidden_dim: int = HIDDEN_DIM,
        latent_dim: int = LATENT_DIM,
        n_tasks: int = N_TASKS,
        n_hosts: int = N_HOSTS,
        temperature: float = TEMPERATURE,
    ):
        super().__init__()
        self.n_tasks = n_tasks
        self.n_hosts = n_hosts
        self.latent_dim = latent_dim
        self.temperature = temperature
        self.hosts = nn.ModuleList([
            nn.Sequential(
                nn.Linear(input_dim, hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, latent_dim),
                nn.ReLU(),
            ) for _ in range(n_hosts)
        ])
        self.transforms = nn.ModuleList([
            nn.Sequential(
                nn.Linear(latent_dim, latent_dim),
                nn.ReLU(),
                nn.Linear(latent_dim, latent_dim),
            ) for _ in range(n_hosts)
        ])
        self.task_emb = nn.Embedding(n_tasks, TASK_EMB_DIM)
        self.router = nn.Sequential(
            nn.Linear(input_dim + TASK_EMB_DIM, ROUTE_HIDDEN_DIM),
            nn.ReLU(),
            nn.Linear(ROUTE_HIDDEN_DIM, n_hosts),
        )
        self.heads = nn.ModuleList([nn.Linear(latent_dim, 2) for _ in range(n_tasks)])

    def route(self, x: torch.Tensor, task_id: int) -> torch.Tensor:
        b = x.shape[0]
        t = torch.full((b,), int(task_id), device=x.device, dtype=torch.long)
        te = self.task_emb(t)
        logits = self.router(torch.cat([x, te], dim=-1))
        return F.softmax(logits / self.temperature, dim=-1)

    def host_stack(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        host_latents = torch.stack([h(x) for h in self.hosts], dim=1)  # [B,H,D]
        transformed = torch.stack([tr(host_latents[:, i, :]) for i, tr in enumerate(self.transforms)], dim=1)
        return host_latents, transformed

    def forward(
        self,
        x: torch.Tensor,
        task_id: int,
        route_override: Optional[torch.Tensor] = None,
        host_mask: Optional[torch.Tensor] = None,
        return_dict: bool = True,
    ) -> Dict[str, torch.Tensor]:
        host_latents, transformed = self.host_stack(x)
        routes = self.route(x, task_id)
        if route_override is not None:
            r = route_override.to(x.device).float()
            if r.ndim == 1:
                r = r.unsqueeze(0).expand(x.shape[0], -1)
            routes = normalize_route_tensor(r)
        if host_mask is not None:
            m = host_mask.to(x.device).float()
            if m.ndim == 1:
                m = m.unsqueeze(0).expand(x.shape[0], -1)
            routes = normalize_route_tensor(routes * m)
        z_fungal = torch.sum(routes.unsqueeze(-1) * transformed, dim=1)
        logits = self.heads[int(task_id)](z_fungal)
        return {
            'logits': logits,
            'routes': routes,
            'z_fungal': z_fungal,
            'host_latents': host_latents,
            'transformed_hosts': transformed,
        }


def entropy_torch(r: torch.Tensor, eps: float = 1e-8) -> torch.Tensor:
    return -torch.sum(r * torch.log(r + eps), dim=-1).mean()


def train_demo_model(seed: int, tasks: List[Dict[str, Any]]) -> DemoFungalMMALS:
    set_seed(seed)
    model = DemoFungalMMALS().to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    for task_id, task in enumerate(tasks):
        loader = make_loader(task['train'], batch_size=BATCH_SIZE, shuffle=True)
        model.train()
        for epoch in range(EPOCHS_PER_TASK):
            for xb, yb in loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                opt.zero_grad(set_to_none=True)
                out = model(xb, task_id)
                ce = F.cross_entropy(out['logits'], yb)
                carbon = 0.002 * entropy_torch(out['routes'])
                metabolic = 1e-4 * (out['z_fungal'].pow(2).mean())
                loss = ce + carbon + metabolic
                loss.backward()
                opt.step()
    model.eval()
    return model

In [ ]:
# ============================================================
# 5. Real RC2O loading hook
# ============================================================

# Edit this cell for the real RC2O architecture.
# Keep the checkpoint frozen: this audit must not train or update it.

def build_frozen_rc2o_model() -> nn.Module:
    """Return an instantiated RC2O model before loading weights.

    TODO for the real run:
        1. Import or paste the exact RC2O model class used to freeze the checkpoint.
        2. Instantiate it with the same architecture/config.
        3. Return the model without training.

    Example pattern:
        from mmals_rc2o_model import MMALS_RC2O_Model
        config = load_config('/content/drive/MyDrive/MMALS/checkpoints/RC2O_config.json')
        model = MMALS_RC2O_Model(config)
        return model
    """
    raise NotImplementedError(
        'Define build_frozen_rc2o_model() for the real RC2O architecture. '
        'For a pipeline smoke test only, set MODEL_SOURCE="demo_smoke", RUN_MODE="smoke", ALLOW_DEMO_SMOKE=True.'
    )


def _extract_state_dict(payload: Any) -> Dict[str, torch.Tensor]:
    """Robustly extract a PyTorch state_dict from common checkpoint payloads."""
    if isinstance(payload, dict):
        for key in ['model_state_dict', 'state_dict', 'model', 'net']:
            if key in payload and isinstance(payload[key], dict):
                return payload[key]
        # It may already be a state dict.
        if all(isinstance(k, str) for k in payload.keys()):
            return payload
    return payload


def load_frozen_checkpoint(path: str | Path) -> nn.Module:
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(
            f'Frozen checkpoint not found: {path}. Mount Drive and update FROZEN_CKPT_PATH or SEED_CKPT_PATHS.'
        )
    model = build_frozen_rc2o_model()
    payload = torch.load(path, map_location=DEVICE)
    state_dict = _extract_state_dict(payload)
    missing, unexpected = model.load_state_dict(state_dict, strict=False)
    if missing:
        print('WARNING missing checkpoint keys:', missing[:20], '...' if len(missing) > 20 else '')
    if unexpected:
        print('WARNING unexpected checkpoint keys:', unexpected[:20], '...' if len(unexpected) > 20 else '')
    model.to(DEVICE).eval()
    for p in model.parameters():
        p.requires_grad = False
    return model


class MMALSForwardAdapter:
    """Adapter around MMALS-like models.

    For the demo model this works directly. For real RC2O, either make the model's forward
    return the contract dictionary, or customize this adapter.
    """
    def __init__(self, model: nn.Module):
        self.model = model

    def forward(
        self,
        x: torch.Tensor,
        task_id: int,
        route_override: Optional[torch.Tensor] = None,
        host_mask: Optional[torch.Tensor] = None,
        require_hosts: bool = True,
    ) -> Dict[str, torch.Tensor]:
        # Preferred path: model supports route_override / host_mask / return_dict.
        try:
            out = self.model(x, task_id, route_override=route_override, host_mask=host_mask, return_dict=True)
            return self._standardize(out, x, task_id)
        except TypeError:
            pass

        # Common path: model supports x, task_id and returns dict/tuple.
        out = self.model(x, task_id)
        out = self._standardize(out, x, task_id)

        # Post-hoc intervention fallback: if transformed_hosts and heads exist, recompute z/logits.
        if route_override is not None or host_mask is not None:
            if 'transformed_hosts' not in out or out['transformed_hosts'] is None:
                raise RuntimeError('Intervention requires transformed_hosts or native model intervention support.')
            routes = out['routes']
            if route_override is not None:
                r = route_override.to(x.device).float()
                if r.ndim == 1:
                    r = r.unsqueeze(0).expand(x.shape[0], -1)
                routes = normalize_route_tensor(r)
            if host_mask is not None:
                m = host_mask.to(x.device).float()
                if m.ndim == 1:
                    m = m.unsqueeze(0).expand(x.shape[0], -1)
                routes = normalize_route_tensor(routes * m)
            z = torch.sum(routes.unsqueeze(-1) * out['transformed_hosts'], dim=1)
            logits = self._head_from_z(z, task_id)
            out['routes'] = routes
            out['z_fungal'] = z
            out['logits'] = logits
        return out

    def _head_from_z(self, z: torch.Tensor, task_id: int) -> torch.Tensor:
        if hasattr(self.model, 'heads'):
            return self.model.heads[int(task_id)](z)
        if hasattr(self.model, 'forward_from_z'):
            return self.model.forward_from_z(z, task_id)
        raise RuntimeError('Cannot compute logits from z_fungal. Add model.heads or forward_from_z.')

    def _standardize(self, out: Any, x: torch.Tensor, task_id: int) -> Dict[str, torch.Tensor]:
        if isinstance(out, dict):
            d = dict(out)
        elif isinstance(out, tuple):
            # Supported tuple conventions:
            # (logits, routes)
            # (logits, routes, z_fungal)
            # (logits, routes, z_fungal, transformed_hosts)
            # (logits, routes, z_fungal, host_latents, transformed_hosts)
            keys = ['logits', 'routes', 'z_fungal', 'host_latents', 'transformed_hosts']
            d = {k: v for k, v in zip(keys, out)}
        else:
            raise TypeError('Model output must be dict or tuple for the adapter.')

        aliases = {
            'route': 'routes',
            'r': 'routes',
            'zf': 'z_fungal',
            'z_f': 'z_fungal',
            'z': 'z_fungal',
            'host_z': 'host_latents',
            'hosts_z': 'host_latents',
            'z_hosts': 'transformed_hosts',
            'transformed_z': 'transformed_hosts',
            'z_tilde': 'transformed_hosts',
        }
        for old, new in aliases.items():
            if old in d and new not in d:
                d[new] = d[old]

        required = ['logits', 'routes']
        missing = [k for k in required if k not in d or d[k] is None]
        if missing:
            raise KeyError(f'Missing required adapter keys: {missing}. Available keys: {list(d.keys())}')

        routes = d['routes']
        if routes.ndim == 1:
            routes = routes.unsqueeze(0).expand(x.shape[0], -1)
        d['routes'] = routes.float()

        if 'z_fungal' not in d or d['z_fungal'] is None:
            if 'transformed_hosts' in d and d['transformed_hosts'] is not None:
                d['z_fungal'] = torch.sum(d['routes'].unsqueeze(-1) * d['transformed_hosts'], dim=1)
            else:
                d['z_fungal'] = torch.empty((x.shape[0], 0), device=x.device)
        return d


def checkpoint_for_seed(seed: int) -> str:
    if MODEL_SOURCE != 'checkpoint':
        return ''
    if isinstance(SEED_CKPT_PATHS, dict) and seed in SEED_CKPT_PATHS:
        return SEED_CKPT_PATHS[seed]
    return FROZEN_CKPT_PATH


def get_audit_model(seed: int, tasks: List[Dict[str, Any]]) -> nn.Module:
    if MODEL_SOURCE == 'checkpoint':
        model = load_frozen_checkpoint(checkpoint_for_seed(seed))
    elif MODEL_SOURCE == 'demo_smoke':
        warnings.warn('MODEL_SOURCE=demo_smoke: this verifies the audit only; it is not RC2O evidence.')
        model = train_demo_model(seed, tasks)
    else:
        raise ValueError("MODEL_SOURCE must be 'checkpoint' or 'demo_smoke'")
    model.eval()
    for p in model.parameters():
        p.requires_grad = False
    return model


## Evaluation core

The functions below compute the normal frozen-policy evaluation and collect the tensors needed for interventions.

Important: this is an audit. Every evaluation function is decorated with `torch.no_grad()` and the model is kept in `eval()` mode.

In [ ]:
# ============================================================
# 6. Evaluation and diagnostic collection
# ============================================================

@torch.no_grad()
def evaluate_task(
    adapter: MMALSForwardAdapter,
    task: Dict[str, Any],
    task_id: int,
    route_override: Optional[torch.Tensor] = None,
    host_mask: Optional[torch.Tensor] = None,
    collect: bool = True,
) -> Dict[str, Any]:
    loader = make_loader(task['test'], shuffle=False)
    total = 0
    correct = 0
    ce_sum = 0.0
    logits_list, probs_list, y_list = [], [], []
    routes_list, z_list = [], []
    transformed_list, host_latents_list = [], []

    adapter.model.eval()
    for xb, yb in loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        out = adapter.forward(xb, task_id, route_override=route_override, host_mask=host_mask)
        logits = out['logits']
        ce = F.cross_entropy(logits, yb, reduction='sum')
        pred = logits.argmax(dim=-1)
        correct += int((pred == yb).sum().item())
        total += int(yb.numel())
        ce_sum += float(ce.item())
        if collect:
            logits_list.append(logits.detach().cpu())
            probs_list.append(F.softmax(logits, dim=-1).detach().cpu())
            y_list.append(yb.detach().cpu())
            routes_list.append(out['routes'].detach().cpu())
            z_list.append(out['z_fungal'].detach().cpu())
            if 'transformed_hosts' in out and out['transformed_hosts'] is not None:
                transformed_list.append(out['transformed_hosts'].detach().cpu())
            if 'host_latents' in out and out['host_latents'] is not None:
                host_latents_list.append(out['host_latents'].detach().cpu())

    result = {
        'task_id': task_id,
        'task_name': task['name'],
        'pair': tuple(task['pair']),
        'n': total,
        'accuracy': correct / max(1, total),
        'ce': ce_sum / max(1, total),
    }
    if collect:
        result.update({
            'logits': torch.cat(logits_list, dim=0),
            'probs': torch.cat(probs_list, dim=0),
            'y': torch.cat(y_list, dim=0),
            'routes': torch.cat(routes_list, dim=0),
            'z_fungal': torch.cat(z_list, dim=0),
            'transformed_hosts': torch.cat(transformed_list, dim=0) if transformed_list else None,
            'host_latents': torch.cat(host_latents_list, dim=0) if host_latents_list else None,
        })
    return result


@torch.no_grad()
def evaluate_all_tasks(adapter: MMALSForwardAdapter, tasks: List[Dict[str, Any]], collect: bool = True) -> Dict[int, Dict[str, Any]]:
    return {task['task_id']: evaluate_task(adapter, task, task['task_id'], collect=collect) for task in tasks}


def summarize_normal_eval(normal: Dict[int, Dict[str, Any]]) -> pd.DataFrame:
    rows = []
    for task_id, res in normal.items():
        routes = tensor_to_numpy(res['routes'])
        row = {
            'task_id': task_id,
            'task_name': res['task_name'],
            'pair': str(res['pair']),
            'accuracy': res['accuracy'],
            'ce': res['ce'],
            'route_entropy_mean': float(entropy_np(routes).mean()),
            'route_entropy_std': float(entropy_np(routes).std()),
            'route_dominance_mean': float(dominance_np(routes).mean()),
            'route_dominance_std': float(dominance_np(routes).std()),
            'dominant_host': int(np.argmax(routes.mean(axis=0))),
        }
        mean_route = routes.mean(axis=0)
        for i, val in enumerate(mean_route):
            row[f'route_host_{i}'] = float(val)
        rows.append(row)
    return pd.DataFrame(rows)

In [ ]:
# ============================================================
# 7. Run frozen baseline evaluation for the first seed
# ============================================================

seed0 = SEEDS[0]
set_seed(seed0)
model = get_audit_model(seed0, tasks)
adapter = MMALSForwardAdapter(model)

normal_eval = evaluate_all_tasks(adapter, tasks, collect=True)
normal_df = summarize_normal_eval(normal_eval)
normal_df.to_csv(TABLES_DIR / 'normal_frozen_eval.csv', index=False)

display(normal_df)
print('Total parameters:', count_parameters(model, trainable_only=False))
print('Model size MB:', round(model_size_mb(model), 3))

In [ ]:
# ============================================================
# 8. Plot route heatmap
# ============================================================

def plot_route_heatmap(normal_df: pd.DataFrame, path: Path) -> None:
    host_cols = [c for c in normal_df.columns if c.startswith('route_host_')]
    mat = normal_df[host_cols].to_numpy(dtype=float)
    fig, ax = plt.subplots(figsize=(8, 4))
    im = ax.imshow(mat, aspect='auto')
    ax.set_title('Axis A — Mean route by context/task')
    ax.set_xlabel('Host')
    ax.set_ylabel('Context / task')
    ax.set_xticks(range(len(host_cols)))
    ax.set_xticklabels([c.replace('route_host_', 'H') for c in host_cols])
    ax.set_yticks(range(len(normal_df)))
    ax.set_yticklabels([f"T{r.task_id}: {r.task_name}" for _, r in normal_df.iterrows()])
    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]):
            ax.text(j, i, f'{mat[i, j]:.2f}', ha='center', va='center', fontsize=8)
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    fig.tight_layout()
    fig.savefig(path, dpi=160)
    plt.show()

plot_route_heatmap(normal_df, PLOTS_DIR / 'route_heatmap.png')

## Test 1 — Host ablation and mutualistic gain

For each task/context `c` and host `i`, the audit removes host `i`, renormalizes the remaining route, and measures:

\[
G_i(c) = \mathcal{L}_{\mathrm{without}\;i}(c) - \mathcal{L}_{\mathrm{full}}(c)
\]

A host is useful for a context when `G_i(c) > 0`, and it is functionally important when ablation also increases CE, lowers accuracy, increases output KL, or shifts the mediated latent state.


In [ ]:
# ============================================================
# 9. Host ablation and mutualistic gain
# ============================================================

@torch.no_grad()
def host_ablation_audit(adapter: MMALSForwardAdapter, tasks: List[Dict[str, Any]], normal: Dict[int, Dict[str, Any]]) -> pd.DataFrame:
    rows = []
    for task in tasks:
        task_id = task['task_id']
        full = normal[task_id]
        full_probs = tensor_to_numpy(full['probs'])
        full_z = full['z_fungal']
        for host_i in range(N_HOSTS):
            mask = torch.ones(N_HOSTS, device=DEVICE)
            mask[host_i] = 0.0
            try:
                abl = evaluate_task(adapter, task, task_id, host_mask=mask, collect=True)
                abl_probs = tensor_to_numpy(abl['probs'])
                output_kl = float(kl_probs_np(full_probs, abl_probs).mean())
                if full_z.numel() > 0 and abl['z_fungal'].numel() > 0:
                    latent_mse = float(F.mse_loss(abl['z_fungal'].float(), full_z.float()).item())
                else:
                    latent_mse = float('nan')
                mean_route = tensor_to_numpy(full['routes']).mean(axis=0)
                route_weight = float(mean_route[host_i])
                ce_increase = float(abl['ce'] - full['ce'])
                acc_drop = float(full['accuracy'] - abl['accuracy'])
                rows.append({
                    'task_id': task_id,
                    'task_name': task['name'],
                    'host': host_i,
                    'route_weight': route_weight,
                    'full_accuracy': full['accuracy'],
                    'ablated_accuracy': abl['accuracy'],
                    'accuracy_drop': acc_drop,
                    'full_ce': full['ce'],
                    'ablated_ce': abl['ce'],
                    'ce_increase': ce_increase,
                    'mutualistic_gain_G': ce_increase,
                    'output_kl': output_kl,
                    'latent_mse': latent_mse,
                    'positive_gain': bool(ce_increase > GAIN_POSITIVE_EPS),
                    'important_acc_drop': bool(acc_drop > ACC_DROP_IMPORTANT),
                    'important_ce_increase': bool(ce_increase > CE_INCREASE_IMPORTANT),
                    'important_output_kl': bool(output_kl > OUTPUT_KL_IMPORTANT),
                    'important_latent_drift': bool(np.isfinite(latent_mse) and latent_mse > LATENT_DRIFT_IMPORTANT),
                })
            except Exception as e:
                rows.append({
                    'task_id': task_id,
                    'task_name': task['name'],
                    'host': host_i,
                    'route_weight': float(tensor_to_numpy(full['routes']).mean(axis=0)[host_i]),
                    'error': repr(e),
                })
    return pd.DataFrame(rows)

ablation_df = host_ablation_audit(adapter, tasks, normal_eval)
ablation_df.to_csv(TABLES_DIR / 'host_ablation_and_mutualistic_gain.csv', index=False)
display(ablation_df.head(20))

In [ ]:
# ============================================================
# 10. Ablation and gain plots
# ============================================================

def plot_metric_heatmap(df: pd.DataFrame, metric: str, path: Path, title: str) -> None:
    if metric not in df.columns:
        print(f'Metric {metric} not available; skipping plot.')
        return
    pivot = df.pivot(index='task_id', columns='host', values=metric).sort_index()
    mat = pivot.to_numpy(dtype=float)
    fig, ax = plt.subplots(figsize=(8, 4))
    im = ax.imshow(mat, aspect='auto')
    ax.set_title(title)
    ax.set_xlabel('Host')
    ax.set_ylabel('Context / task')
    ax.set_xticks(range(mat.shape[1]))
    ax.set_xticklabels([f'H{i}' for i in pivot.columns])
    ax.set_yticks(range(mat.shape[0]))
    ax.set_yticklabels([f'T{i}' for i in pivot.index])
    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]):
            val = mat[i, j]
            label = 'nan' if not np.isfinite(val) else f'{val:.3f}'
            ax.text(j, i, label, ha='center', va='center', fontsize=8)
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    fig.tight_layout()
    fig.savefig(path, dpi=160)
    plt.show()

plot_metric_heatmap(ablation_df, 'accuracy_drop', PLOTS_DIR / 'host_ablation_accuracy_drop.png', 'Axis A — Accuracy drop under host ablation')
plot_metric_heatmap(ablation_df, 'mutualistic_gain_G', PLOTS_DIR / 'mutualistic_gain_heatmap.png', 'Axis A — Mutualistic gain G = CE_without - CE_full')
plot_metric_heatmap(ablation_df, 'output_kl', PLOTS_DIR / 'host_ablation_output_kl.png', 'Axis A — Output KL drift under host ablation')

## Test 2 — Host/context route swap

Ablation tells us whether a host matters. Swap tells us whether a host-route pattern is **context-specific**.

For tasks \(a\) and \(b\), we evaluate task \(a\) using the mean route learned for task \(b\), and vice versa.

If the swap causes a strong loss increase or accuracy collapse, the route-function binding is non-arbitrary.

In [ ]:
# ============================================================
# 11. Host route swap audit
# ============================================================

@torch.no_grad()
def route_swap_audit(adapter: MMALSForwardAdapter, tasks: List[Dict[str, Any]], normal: Dict[int, Dict[str, Any]]) -> pd.DataFrame:
    mean_routes = {
        tid: normal[tid]['routes'].float().mean(dim=0).to(DEVICE)
        for tid in normal.keys()
    }
    rows = []
    for task_a, task_b in itertools.permutations(tasks, 2):
        a = task_a['task_id']
        b = task_b['task_id']
        full = normal[a]
        try:
            swapped = evaluate_task(adapter, task_a, a, route_override=mean_routes[b], collect=True)
            full_probs = tensor_to_numpy(full['probs'])
            swap_probs = tensor_to_numpy(swapped['probs'])
            output_kl = float(kl_probs_np(full_probs, swap_probs).mean())
            if full['z_fungal'].numel() > 0 and swapped['z_fungal'].numel() > 0:
                latent_mse = float(F.mse_loss(swapped['z_fungal'].float(), full['z_fungal'].float()).item())
            else:
                latent_mse = float('nan')
            rows.append({
                'eval_task_id': a,
                'eval_task_name': task_a['name'],
                'route_source_task_id': b,
                'route_source_task_name': task_b['name'],
                'full_accuracy': full['accuracy'],
                'swapped_accuracy': swapped['accuracy'],
                'accuracy_drop': float(full['accuracy'] - swapped['accuracy']),
                'full_ce': full['ce'],
                'swapped_ce': swapped['ce'],
                'ce_increase': float(swapped['ce'] - full['ce']),
                'output_kl': output_kl,
                'latent_mse': latent_mse,
            })
        except Exception as e:
            rows.append({
                'eval_task_id': a,
                'eval_task_name': task_a['name'],
                'route_source_task_id': b,
                'route_source_task_name': task_b['name'],
                'error': repr(e),
            })
    return pd.DataFrame(rows)

swap_df = route_swap_audit(adapter, tasks, normal_eval)
swap_df.to_csv(TABLES_DIR / 'host_route_swap_audit.csv', index=False)
display(swap_df.head(20))

In [ ]:
# ============================================================
# 12. Swap heatmap
# ============================================================

def plot_swap_heatmap(swap_df: pd.DataFrame, metric: str, path: Path) -> None:
    if metric not in swap_df.columns:
        print(f'Metric {metric} not available; skipping swap plot.')
        return
    pivot = swap_df.pivot(index='eval_task_id', columns='route_source_task_id', values=metric).sort_index()
    mat = pivot.to_numpy(dtype=float)
    fig, ax = plt.subplots(figsize=(7, 5))
    im = ax.imshow(mat, aspect='auto')
    ax.set_title(f'Axis A — Route swap {metric}')
    ax.set_xlabel('Route source context')
    ax.set_ylabel('Evaluated context')
    ax.set_xticks(range(mat.shape[1]))
    ax.set_xticklabels([f'T{i}' for i in pivot.columns])
    ax.set_yticks(range(mat.shape[0]))
    ax.set_yticklabels([f'T{i}' for i in pivot.index])
    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]):
            val = mat[i, j]
            label = 'nan' if not np.isfinite(val) else f'{val:.3f}'
            ax.text(j, i, label, ha='center', va='center', fontsize=8)
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    fig.tight_layout()
    fig.savefig(path, dpi=160)
    plt.show()

plot_swap_heatmap(swap_df, 'accuracy_drop', PLOTS_DIR / 'route_swap_accuracy_drop.png')
plot_swap_heatmap(swap_df, 'ce_increase', PLOTS_DIR / 'route_swap_ce_increase.png')

## Test 3 — Collapse controls

The collapse controls test non-triviality:

1. **Uniform route control:** every host is used equally.
2. **Shuffled route control:** learned route vectors are reassigned to the wrong contexts.
3. **Single global host control:** all tasks are forced through the best global host by average route/usefulness.

If frozen RC2O outperforms these controls, specialization is less likely to be a cosmetic artifact.

In [ ]:
# ============================================================
# 13. Collapse controls
# ============================================================

@torch.no_grad()
def collapse_controls_audit(adapter: MMALSForwardAdapter, tasks: List[Dict[str, Any]], normal: Dict[int, Dict[str, Any]], ablation_df: pd.DataFrame) -> pd.DataFrame:
    mean_routes = {
        tid: normal[tid]['routes'].float().mean(dim=0).to(DEVICE)
        for tid in normal.keys()
    }
    # Single global host: choose by average route weight plus positive gain where available.
    route_scores = np.zeros(N_HOSTS, dtype=float)
    for tid, r in mean_routes.items():
        route_scores += tensor_to_numpy(r)
    if 'mutualistic_gain_G' in ablation_df.columns:
        gain_scores = ablation_df.groupby('host')['mutualistic_gain_G'].mean().reindex(range(N_HOSTS)).fillna(0).to_numpy()
    else:
        gain_scores = np.zeros(N_HOSTS, dtype=float)
    combined = route_scores + np.maximum(0, gain_scores)
    best_global_host = int(np.argmax(combined))

    rows = []
    uniform_route = torch.ones(N_HOSTS, device=DEVICE) / N_HOSTS
    rng = np.random.default_rng(123)
    task_ids = [t['task_id'] for t in tasks]
    shuffled_ids = task_ids.copy()
    rng.shuffle(shuffled_ids)
    if shuffled_ids == task_ids and len(shuffled_ids) > 1:
        shuffled_ids = shuffled_ids[1:] + shuffled_ids[:1]

    controls = []
    controls.append(('uniform_route', lambda tid: uniform_route, None))
    controls.append(('shuffled_route', lambda tid: mean_routes[shuffled_ids[task_ids.index(tid)]], None))
    mask = torch.zeros(N_HOSTS, device=DEVICE)
    mask[best_global_host] = 1.0
    controls.append((f'single_global_host_{best_global_host}', lambda tid: None, mask))

    for control_name, route_fn, mask_const in controls:
        for task in tasks:
            tid = task['task_id']
            full = normal[tid]
            try:
                if mask_const is not None:
                    res = evaluate_task(adapter, task, tid, host_mask=mask_const, collect=True)
                else:
                    res = evaluate_task(adapter, task, tid, route_override=route_fn(tid), collect=True)
                rows.append({
                    'control': control_name,
                    'task_id': tid,
                    'task_name': task['name'],
                    'full_accuracy': full['accuracy'],
                    'control_accuracy': res['accuracy'],
                    'accuracy_drop': float(full['accuracy'] - res['accuracy']),
                    'full_ce': full['ce'],
                    'control_ce': res['ce'],
                    'ce_increase': float(res['ce'] - full['ce']),
                })
            except Exception as e:
                rows.append({
                    'control': control_name,
                    'task_id': tid,
                    'task_name': task['name'],
                    'error': repr(e),
                })
    return pd.DataFrame(rows)

collapse_df = collapse_controls_audit(adapter, tasks, normal_eval, ablation_df)
collapse_df.to_csv(TABLES_DIR / 'collapse_controls.csv', index=False)
display(collapse_df)

In [ ]:
# ============================================================
# 14. Collapse control summary plot
# ============================================================

def plot_collapse_summary(collapse_df: pd.DataFrame, path: Path) -> None:
    if 'accuracy_drop' not in collapse_df.columns:
        print('accuracy_drop unavailable; skipping collapse summary plot.')
        return
    summary = collapse_df.groupby('control', as_index=False).agg(
        mean_accuracy_drop=('accuracy_drop', 'mean'),
        mean_ce_increase=('ce_increase', 'mean'),
    )
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.bar(summary['control'], summary['mean_accuracy_drop'])
    ax.set_title('Axis A — Mean accuracy drop under collapse controls')
    ax.set_ylabel('Mean accuracy drop')
    ax.set_xticks(range(len(summary)))
    ax.set_xticklabels(summary['control'], rotation=30, ha='right')
    fig.tight_layout()
    fig.savefig(path, dpi=160)
    plt.show()
    display(summary)

plot_collapse_summary(collapse_df, PLOTS_DIR / 'collapse_controls_accuracy_drop_summary.png')

## Test 4 — Representation separation

This section asks whether hosts compute distinguishable latent transformations.

The main metric is linear CKA between host transformed features within each context. If all hosts are redundant copies, CKA is high and separation is low.

A simple separation score is:

\[
Separation(c) = 1 - mean_{i \ne j}\ CKA(Z_i^c, Z_j^c)
\]

This is not a proof of causality by itself. It becomes meaningful when combined with routing, gain, and ablation evidence.

In [ ]:
# ============================================================
# 15. Representation separation metrics
# ============================================================

def center_matrix(x: torch.Tensor) -> torch.Tensor:
    return x - x.mean(dim=0, keepdim=True)


def linear_cka_torch(x: torch.Tensor, y: torch.Tensor, eps: float = 1e-12) -> float:
    x = center_matrix(x.float())
    y = center_matrix(y.float())
    hsic = torch.norm(x.T @ y, p='fro') ** 2
    norm_x = torch.norm(x.T @ x, p='fro')
    norm_y = torch.norm(y.T @ y, p='fro')
    denom = torch.clamp(norm_x * norm_y, min=eps)
    return float((hsic / denom).detach().cpu().item())


def cosine_centroid_separation(features: torch.Tensor, eps: float = 1e-8) -> Tuple[float, np.ndarray]:
    # features: [B,H,D]
    centroids = features.float().mean(dim=0)  # [H,D]
    centroids = F.normalize(centroids, dim=-1, eps=eps)
    sim = centroids @ centroids.T
    h = sim.shape[0]
    off = sim[~torch.eye(h, dtype=torch.bool)]
    separation = 1.0 - float(off.mean().detach().cpu().item())
    return separation, tensor_to_numpy(sim)


def representation_separation_audit(normal: Dict[int, Dict[str, Any]]) -> Tuple[pd.DataFrame, Dict[int, np.ndarray]]:
    rows = []
    cka_mats = {}
    for task_id, res in normal.items():
        feats = res.get('transformed_hosts', None)
        if feats is None:
            feats = res.get('host_latents', None)
        if feats is None:
            rows.append({
                'task_id': task_id,
                'task_name': res['task_name'],
                'error': 'No transformed_hosts or host_latents available.'
            })
            continue
        # Limit for CKA memory safety.
        max_n = min(512, feats.shape[0])
        feats_small = feats[:max_n].float()
        h = feats_small.shape[1]
        cka = np.eye(h, dtype=float)
        for i in range(h):
            for j in range(i + 1, h):
                val = linear_cka_torch(feats_small[:, i, :], feats_small[:, j, :])
                cka[i, j] = cka[j, i] = val
        off_diag = cka[~np.eye(h, dtype=bool)]
        cka_sep = 1.0 - float(np.mean(off_diag))
        centroid_sep, centroid_sim = cosine_centroid_separation(feats_small)
        cka_mats[task_id] = cka
        rows.append({
            'task_id': task_id,
            'task_name': res['task_name'],
            'mean_offdiag_cka': float(np.mean(off_diag)),
            'cka_separation': cka_sep,
            'centroid_cosine_separation': centroid_sep,
            'representation_separated': bool(cka_sep > CKA_SEPARATION_MIN),
        })
    return pd.DataFrame(rows), cka_mats

representation_df, cka_mats = representation_separation_audit(normal_eval)
representation_df.to_csv(TABLES_DIR / 'representation_separation.csv', index=False)
display(representation_df)

In [ ]:
# ============================================================
# 16. CKA plots
# ============================================================

def plot_cka_mats(cka_mats: Dict[int, np.ndarray], path_prefix: Path) -> None:
    for task_id, mat in cka_mats.items():
        fig, ax = plt.subplots(figsize=(5, 4))
        im = ax.imshow(mat, vmin=0, vmax=1)
        ax.set_title(f'Axis A — Host CKA similarity, task {task_id}')
        ax.set_xlabel('Host')
        ax.set_ylabel('Host')
        ax.set_xticks(range(mat.shape[1]))
        ax.set_yticks(range(mat.shape[0]))
        ax.set_xticklabels([f'H{i}' for i in range(mat.shape[1])])
        ax.set_yticklabels([f'H{i}' for i in range(mat.shape[0])])
        for i in range(mat.shape[0]):
            for j in range(mat.shape[1]):
                ax.text(j, i, f'{mat[i, j]:.2f}', ha='center', va='center', fontsize=8)
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        fig.tight_layout()
        out = path_prefix.parent / f'{path_prefix.name}_task_{task_id}.png'
        fig.savefig(out, dpi=160)
        plt.show()

plot_cka_mats(cka_mats, PLOTS_DIR / 'host_cka_similarity')

## Host Specialization Index and claim table

The audit does not let a single metric decide the claim.

A host-context pair is considered specialized only when several signals agree:

- routed with meaningful weight,
- route is concentrated at the context level,
- mutualistic gain is positive,
- ablation causes performance or behavior drift,
- representation separation is present.

The index below is a transparent score for ranking; the claim table remains rule-based and conservative.

In [ ]:
# ============================================================
# 17. Host Specialization Index and rule-based claim table
# ============================================================

def minmax_series(s: pd.Series) -> pd.Series:
    s = pd.to_numeric(s, errors='coerce')
    lo, hi = s.min(), s.max()
    if not np.isfinite(lo) or not np.isfinite(hi) or abs(hi - lo) < 1e-12:
        return pd.Series(np.zeros(len(s)), index=s.index)
    return (s - lo) / (hi - lo)


def build_axis_a_claim_table(
    normal_df: pd.DataFrame,
    ablation_df: pd.DataFrame,
    representation_df: pd.DataFrame,
) -> pd.DataFrame:
    rows = []
    rep_map = representation_df.set_index('task_id').to_dict(orient='index') if 'task_id' in representation_df.columns else {}
    norm_map = normal_df.set_index('task_id').to_dict(orient='index')

    df = ablation_df.copy()
    for col in ['mutualistic_gain_G', 'accuracy_drop', 'ce_increase', 'output_kl', 'latent_mse', 'route_weight']:
        if col not in df.columns:
            df[col] = np.nan
        df[f'{col}_norm'] = minmax_series(df[col])

    for _, r in df.iterrows():
        task_id = int(r['task_id'])
        host = int(r['host'])
        nrow = norm_map.get(task_id, {})
        rep = rep_map.get(task_id, {})
        route_entropy = float(nrow.get('route_entropy_mean', np.nan))
        route_dominance = float(nrow.get('route_dominance_mean', np.nan))
        dominant_host = int(nrow.get('dominant_host', -1))
        rep_sep = float(rep.get('cka_separation', np.nan))

        routing_specialized = bool(np.isfinite(route_entropy) and np.isfinite(route_dominance) and route_entropy < ENTROPY_SPECIALIZED_MAX and route_dominance > DOMINANCE_SPECIALIZED_MIN)
        routed_host = bool(float(r.get('route_weight', 0.0)) > (1.0 / N_HOSTS) * 0.75 or host == dominant_host)
        useful_gain = bool(float(r.get('mutualistic_gain_G', 0.0)) > GAIN_POSITIVE_EPS)
        ablation_sensitive = bool(
            float(r.get('accuracy_drop', 0.0)) > ACC_DROP_IMPORTANT or
            float(r.get('ce_increase', 0.0)) > CE_INCREASE_IMPORTANT or
            float(r.get('output_kl', 0.0)) > OUTPUT_KL_IMPORTANT or
            (np.isfinite(float(r.get('latent_mse', np.nan))) and float(r.get('latent_mse', 0.0)) > LATENT_DRIFT_IMPORTANT)
        )
        representation_separated = bool(np.isfinite(rep_sep) and rep_sep > CKA_SEPARATION_MIN)

        # Transparent ranking score, not the sole decision criterion.
        hsi = (
            0.25 * float(r.get('route_weight_norm', 0.0)) +
            0.25 * max(0.0, float(r.get('mutualistic_gain_G_norm', 0.0))) +
            0.20 * max(0.0, float(r.get('accuracy_drop_norm', 0.0))) +
            0.15 * max(0.0, float(r.get('output_kl_norm', 0.0))) +
            0.15 * max(0.0, rep_sep if np.isfinite(rep_sep) else 0.0)
        )

        if routing_specialized and routed_host and useful_gain and ablation_sensitive and representation_separated:
            claim_level = 'functional_host_specialization'
        elif routing_specialized and routed_host and useful_gain and ablation_sensitive:
            claim_level = 'functional_contribution_without_representation_proof'
        elif routing_specialized and routed_host and useful_gain:
            claim_level = 'useful_routed_host'
        elif routing_specialized and routed_host:
            claim_level = 'routing_specialization_only'
        else:
            claim_level = 'no_specialization_claim'

        rows.append({
            'task_id': task_id,
            'task_name': r.get('task_name', ''),
            'host': host,
            'route_weight': float(r.get('route_weight', np.nan)),
            'dominant_host': dominant_host,
            'route_entropy_mean': route_entropy,
            'route_dominance_mean': route_dominance,
            'mutualistic_gain_G': float(r.get('mutualistic_gain_G', np.nan)),
            'accuracy_drop': float(r.get('accuracy_drop', np.nan)),
            'ce_increase': float(r.get('ce_increase', np.nan)),
            'output_kl': float(r.get('output_kl', np.nan)),
            'latent_mse': float(r.get('latent_mse', np.nan)),
            'cka_separation': rep_sep,
            'routing_specialized': routing_specialized,
            'routed_host': routed_host,
            'useful_gain': useful_gain,
            'ablation_sensitive': ablation_sensitive,
            'representation_separated': representation_separated,
            'host_specialization_index': float(hsi),
            'claim_level': claim_level,
        })
    return pd.DataFrame(rows).sort_values(['task_id', 'host_specialization_index'], ascending=[True, False])

claim_df = build_axis_a_claim_table(normal_df, ablation_df, representation_df)
claim_df.to_csv(TABLES_DIR / 'axis_a_claim_table.csv', index=False)
display(claim_df)

In [ ]:
# ============================================================
# 18. Claim summary plot
# ============================================================

def plot_claim_hsi(claim_df: pd.DataFrame, path: Path) -> None:
    pivot = claim_df.pivot(index='task_id', columns='host', values='host_specialization_index').sort_index()
    mat = pivot.to_numpy(dtype=float)
    fig, ax = plt.subplots(figsize=(8, 4))
    im = ax.imshow(mat, aspect='auto')
    ax.set_title('Axis A — Host Specialization Index')
    ax.set_xlabel('Host')
    ax.set_ylabel('Context / task')
    ax.set_xticks(range(mat.shape[1]))
    ax.set_xticklabels([f'H{i}' for i in pivot.columns])
    ax.set_yticks(range(mat.shape[0]))
    ax.set_yticklabels([f'T{i}' for i in pivot.index])
    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]):
            ax.text(j, i, f'{mat[i, j]:.2f}', ha='center', va='center', fontsize=8)
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    fig.tight_layout()
    fig.savefig(path, dpi=160)
    plt.show()

plot_claim_hsi(claim_df, PLOTS_DIR / 'host_specialization_index_heatmap.png')

summary_claims = claim_df.groupby('claim_level', as_index=False).size().sort_values('size', ascending=False)
display(summary_claims)
summary_claims.to_csv(TABLES_DIR / 'axis_a_claim_summary.csv', index=False)

## Cross-seed specialization stability

Host IDs may permute across seeds. Therefore, cross-seed comparison should align hosts before judging stability.

This notebook aligns hosts using the Hungarian algorithm on either:

1. route profiles, or
2. gain profiles.

A robust result is not necessarily “host 2 always specializes in task 3.” A better claim is:

> Equivalent host functions recur across seeds after host-label alignment.

In [ ]:
# ============================================================
# 19. Cross-seed audit helpers and optional full cross-seed run
# ============================================================

def host_profile_matrix_from_claims(claim_df: pd.DataFrame, value_col: str = 'host_specialization_index') -> np.ndarray:
    # Shape [tasks, hosts] -> profile per host is [tasks]
    pivot = claim_df.pivot(index='task_id', columns='host', values=value_col).sort_index().fillna(0.0)
    return pivot.to_numpy(dtype=float)


def align_hosts(reference: np.ndarray, candidate: np.ndarray) -> Tuple[List[int], float]:
    """Align candidate host columns to reference host columns.

    reference and candidate have shape [tasks, hosts]. Returns permutation p where
    candidate[:, p[j]] corresponds to reference[:, j].
    """
    h = reference.shape[1]
    # Similarity by cosine between host profiles.
    ref = reference.copy()
    cand = candidate.copy()
    ref = ref / (np.linalg.norm(ref, axis=0, keepdims=True) + 1e-12)
    cand = cand / (np.linalg.norm(cand, axis=0, keepdims=True) + 1e-12)
    sim = ref.T @ cand  # [H,H]
    cost = -sim
    if SCIPY_AVAILABLE:
        row_ind, col_ind = linear_sum_assignment(cost)
        perm = [None] * h
        for r, c in zip(row_ind, col_ind):
            perm[r] = int(c)
    else:
        best_score = -1e18
        best_perm = list(range(h))
        for perm_candidate in itertools.permutations(range(h)):
            score = sum(sim[j, perm_candidate[j]] for j in range(h))
            if score > best_score:
                best_score = score
                best_perm = list(perm_candidate)
        perm = best_perm
    aligned_score = float(np.mean([sim[j, perm[j]] for j in range(h)]))
    return perm, aligned_score


def summarize_cross_seed_from_claim_tables(seed_claim_tables: Dict[int, pd.DataFrame]) -> pd.DataFrame:
    seeds = sorted(seed_claim_tables.keys())
    if len(seeds) < 2:
        return pd.DataFrame([{'status': 'skipped', 'reason': 'Need at least two seed claim tables.'}])
    ref_seed = seeds[0]
    ref_mat = host_profile_matrix_from_claims(seed_claim_tables[ref_seed])
    rows = []
    for seed in seeds[1:]:
        cand_mat = host_profile_matrix_from_claims(seed_claim_tables[seed])
        perm, score = align_hosts(ref_mat, cand_mat)
        aligned = cand_mat[:, perm]
        corr = np.corrcoef(ref_mat.ravel(), aligned.ravel())[0, 1] if ref_mat.size and np.std(aligned) > 1e-12 and np.std(ref_mat) > 1e-12 else np.nan
        rows.append({
            'reference_seed': ref_seed,
            'candidate_seed': seed,
            'host_permutation_candidate_to_reference_order': str(perm),
            'mean_aligned_cosine': score,
            'flattened_profile_correlation': float(corr) if np.isfinite(corr) else np.nan,
        })
    return pd.DataFrame(rows)


def run_axis_a_for_seed(seed: int) -> pd.DataFrame:
    """Run the full Axis A audit for one seed and export seed-specific tables."""
    print(f'Running full Axis A audit for seed {seed}')
    clear_memory()
    seed_tasks = build_tasks(DATASET_NAME, PAIR_MODE, seed=42)
    seed_model = get_audit_model(seed, seed_tasks)
    seed_adapter = MMALSForwardAdapter(seed_model)
    seed_normal = evaluate_all_tasks(seed_adapter, seed_tasks, collect=True)
    seed_normal_df = summarize_normal_eval(seed_normal)
    seed_ablation_df = host_ablation_audit(seed_adapter, seed_tasks, seed_normal)
    seed_rep_df, _ = representation_separation_audit(seed_normal)
    seed_claim_df = build_axis_a_claim_table(seed_normal_df, seed_ablation_df, seed_rep_df)
    seed_normal_df.to_csv(TABLES_DIR / f'normal_frozen_eval_seed_{seed}.csv', index=False)
    seed_ablation_df.to_csv(TABLES_DIR / f'host_ablation_and_mutualistic_gain_seed_{seed}.csv', index=False)
    seed_rep_df.to_csv(TABLES_DIR / f'representation_separation_seed_{seed}.csv', index=False)
    seed_claim_df.to_csv(TABLES_DIR / f'axis_a_claim_table_seed_{seed}.csv', index=False)
    return seed_claim_df


# Cross-seed evidence is part of the real RC2O workflow.
# It runs automatically when:
# - there are at least 2 seeds,
# - MODEL_SOURCE='checkpoint', and
# - either a per-seed checkpoint map is supplied or FROZEN_CKPT_PATH is intentionally reused.
RUN_CROSS_SEED_FULL_AUDIT = (MODEL_SOURCE == 'checkpoint' and len(SEEDS) >= 2)

seed_claim_tables = {seed0: claim_df}

if RUN_CROSS_SEED_FULL_AUDIT:
    if not SEED_CKPT_PATHS:
        print('WARNING: SEED_CKPT_PATHS is empty. The notebook will reuse FROZEN_CKPT_PATH for each seed.')
        print('For true cross-seed evidence, provide a frozen checkpoint per seed when available.')
    for seed in SEEDS[1:]:
        seed_claim_tables[seed] = run_axis_a_for_seed(seed)
else:
    print('Cross-seed full audit not run. Set MODEL_SOURCE="checkpoint" and provide at least two seeds for real stability evidence.')

cross_seed_df = summarize_cross_seed_from_claim_tables(seed_claim_tables)
cross_seed_df.to_csv(TABLES_DIR / 'cross_seed_specialization_alignment.csv', index=False)
display(cross_seed_df)


## Manifest and export package

The final artifact bundle contains:

- CSV tables,
- PNG plots,
- JSON manifest,
- ZIP package.

The manifest is part of the freeze discipline: it records the candidate name, run mode, thresholds, dataset regime, checkpoint hash when available, and notebook identity.

In [ ]:
# ============================================================
# 20. Manifest, ZIP export, and Google Drive publication
# ============================================================

created_utc = time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime())
run_tag = f"{NOTEBOOK_NAME}_{RUN_MODE}_{MODEL_SOURCE}_{created_utc.replace(':', '').replace('-', '')}"

manifest = {
    'notebook': NOTEBOOK_NAME,
    'frozen_policy_name': FROZEN_POLICY_NAME,
    'model_source': MODEL_SOURCE,
    'is_rc2o_scientific_evidence': bool(MODEL_SOURCE == 'checkpoint'),
    'frozen_checkpoint_path': FROZEN_CKPT_PATH if MODEL_SOURCE == 'checkpoint' else None,
    'frozen_checkpoint_sha256': file_sha256(FROZEN_CKPT_PATH) if MODEL_SOURCE == 'checkpoint' else None,
    'seed_checkpoint_paths': SEED_CKPT_PATHS if MODEL_SOURCE == 'checkpoint' else {},
    'created_utc': created_utc,
    'run_tag': run_tag,
    'run_mode': RUN_MODE,
    'dataset_name': DATASET_NAME,
    'pair_mode': PAIR_MODE,
    'train_per_class': TRAIN_PER_CLASS,
    'test_per_class': TEST_PER_CLASS,
    'seeds': SEEDS,
    'n_tasks': N_TASKS,
    'n_hosts': N_HOSTS,
    'batch_size': BATCH_SIZE,
    'thresholds': {
        'ENTROPY_SPECIALIZED_MAX': ENTROPY_SPECIALIZED_MAX,
        'DOMINANCE_SPECIALIZED_MIN': DOMINANCE_SPECIALIZED_MIN,
        'GAIN_POSITIVE_EPS': GAIN_POSITIVE_EPS,
        'ACC_DROP_IMPORTANT': ACC_DROP_IMPORTANT,
        'CE_INCREASE_IMPORTANT': CE_INCREASE_IMPORTANT,
        'OUTPUT_KL_IMPORTANT': OUTPUT_KL_IMPORTANT,
        'LATENT_DRIFT_IMPORTANT': LATENT_DRIFT_IMPORTANT,
        'CKA_SEPARATION_MIN': CKA_SEPARATION_MIN,
    },
    'parameter_count_total': count_parameters(model, trainable_only=False),
    'model_size_mb_fp32_estimate': model_size_mb(model),
    'tables': sorted([p.name for p in TABLES_DIR.glob('*.csv')]),
    'plots': sorted([p.name for p in PLOTS_DIR.glob('*.png')]),
    'conservative_claim': (
        'MMALS exhibits measurable functional host specialization only when routing concentration, '
        'mutualistic gain, ablation sensitivity, representation separation, and cross-seed stability agree. '
        'This audit does not claim biological proof.'
    ),
}

manifest_path = RESULTS_DIR / 'axis_a_manifest.json'
with manifest_path.open('w', encoding='utf-8') as f:
    json.dump(manifest, f, indent=2)

# Also save a short markdown report.
report_path = RESULTS_DIR / 'AXIS_A_HOST_SPECIALIZATION_AUDIT_SUMMARY.md'
with report_path.open('w', encoding='utf-8') as f:
    f.write('# Axis A Host Specialization Audit Summary

')
    f.write(f'Frozen policy: **{FROZEN_POLICY_NAME}**

')
    f.write(f'Model source: `{MODEL_SOURCE}`

')
    f.write(f'RC2O scientific evidence: `{MODEL_SOURCE == "checkpoint"}`

')
    f.write(f'Dataset: `{DATASET_NAME}` / `{PAIR_MODE}`

')
    f.write(f'Run mode: `{RUN_MODE}`

')
    f.write('## Claim counts

')
    f.write(summary_claims.to_markdown(index=False))
    f.write('

## Cross-seed alignment

')
    f.write(cross_seed_df.to_markdown(index=False))
    f.write('

## Conservative interpretation

')
    f.write(manifest['conservative_claim'])
    f.write('
')

zip_path = Path(f'./{NOTEBOOK_NAME}_Artifacts.zip')
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for p in RESULTS_DIR.rglob('*'):
        if p.is_file():
            zf.write(p, arcname=str(p.relative_to(RESULTS_DIR.parent)))

print('Manifest:', manifest_path)
print('Report:', report_path)
print('ZIP:', zip_path.resolve())
print('ZIP sha256:', file_sha256(zip_path))


def publish_results_to_drive() -> Optional[Path]:
    """Copy results into Google Drive when running in Colab."""
    if not PUBLISH_TO_DRIVE:
        print('PUBLISH_TO_DRIVE=False; skipping Drive publication.')
        return None
    try:
        from google.colab import drive  # type: ignore
        if not Path('/content/drive/MyDrive').exists():
            drive.mount('/content/drive')
    except Exception as e:
        print('Google Drive publication skipped. This usually means the notebook is not running in Colab.')
        print('Reason:', repr(e))
        return None

    target_dir = Path(DRIVE_OUTPUT_ROOT) / run_tag
    target_dir.mkdir(parents=True, exist_ok=True)

    # Copy the full results folder, ZIP, manifest, and report.
    shutil.copytree(RESULTS_DIR, target_dir / RESULTS_DIR.name, dirs_exist_ok=True)
    shutil.copy2(zip_path, target_dir / zip_path.name)
    shutil.copy2(manifest_path, target_dir / manifest_path.name)
    shutil.copy2(report_path, target_dir / report_path.name)

    # Copy the current notebook if available in /content.
    if COPY_NOTEBOOK_TO_DRIVE:
        candidate_notebooks = [
            Path(f'/content/{NOTEBOOK_NAME}.ipynb'),
            Path(f'/content/{NOTEBOOK_NAME.replace("_UPDATED", "")}.ipynb'),
        ]
        for nb_path in candidate_notebooks:
            if nb_path.exists():
                shutil.copy2(nb_path, target_dir / nb_path.name)
                break

    print('Published Axis A results to Google Drive:')
    print(str(target_dir))
    return target_dir


drive_publish_dir = publish_results_to_drive()


## Optional clean PDF export

The previous PDF export failed because some Markdown math was converted into invalid LaTeX. This notebook avoids that pattern. The optional cell below exports a PDF and then publishes it to the same Drive folder when available.


In [ ]:
# ============================================================
# 21. Optional clean PDF export and Drive copy
# ============================================================
# This cell is optional. It should run after the ZIP/Drive publication cell.
EXPORT_PDF = False

if EXPORT_PDF:
    import subprocess
    nb_path_candidates = [
        Path(f'/content/{NOTEBOOK_NAME}.ipynb'),
        Path(f'/content/{NOTEBOOK_NAME.replace("_UPDATED", "")}.ipynb'),
    ]
    nb_path = next((p for p in nb_path_candidates if p.exists()), None)
    if nb_path is None:
        print('Notebook file not found under /content; skipping PDF export.')
    else:
        cmd = [
            'jupyter', 'nbconvert', '--to', 'pdf',
            '--output-dir', '/content',
            '--PDFExporter.latex_command', "['xelatex', '{filename}', '-interaction=nonstopmode']",
            str(nb_path),
        ]
        print('Running:', ' '.join(cmd))
        completed = subprocess.run(cmd, capture_output=True, text=True)
        print(completed.stdout)
        if completed.returncode != 0:
            print(completed.stderr[-4000:])
            raise RuntimeError('PDF export failed. Check the LaTeX log above.')
        pdf_path = Path('/content') / f'{nb_path.stem}.pdf'
        print('PDF exported:', pdf_path)
        if drive_publish_dir is not None and pdf_path.exists():
            shutil.copy2(pdf_path, Path(drive_publish_dir) / pdf_path.name)
            print('PDF copied to Drive:', Path(drive_publish_dir) / pdf_path.name)


## Final interpretation template

Use this wording after running the real frozen RC2O checkpoint:

> RC2O was frozen as the policy candidate. Axis A performed post-hoc host-specialization diagnostics without modifying routing, selector, guard, loss, or checkpoint weights. The audit supports a functional specialization claim only where routing specialization, positive mutualistic gain, ablation sensitivity, representation separation, and cross-seed stability jointly agree. The claim is architectural and empirical; it is not interpreted as biological proof.

## Evidence ladder

| Evidence pattern | Claim level |
|---|---|
| entropy ↓ and dominance ↑ only | routing specialization only |
| routing + positive mutualistic gain | useful routed host |
| routing + gain + ablation drop | functional contribution |
| plus representation separation | functional host specialization |
| plus cross-seed stability | robust functional specialization |
| plus collapse controls fail | non-trivial specialization beyond route/capacity artifact |

## Drive publication

The final cell publishes the results to:

```text
/content/drive/MyDrive/MMALS/RC2O/Axis_A_Host_Specialization_Audit/<run_tag>/
```

It copies the ZIP artifact, manifest, summary markdown, tables, plots, and the results folder.
